# TransformG1 - Practica ETL Semana 2

Grupo: G1

Integrante: Carlos Diaz

Proyecto: ciberseguridad y monitoreo de incidentes de red. Este notebook documenta limpieza, normalizacion, transformacion y validacion de tres fuentes relacionadas: alertas de red, inventario de activos y catalogo de vulnerabilidades.

## 1. Exploracion inicial de fuentes

Se revisan estructura, filas, columnas, tipos de datos, nulos, duplicados y posibles llaves de relacion. Las columnas principales de relacion son `machine_id`, `attack_type`, `protocol` y `severity_level`.

In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW_DIR = Path('data/raw')
alerts_raw = pd.read_csv(RAW_DIR / 'cybersecurity_attacks.csv')
assets_raw = pd.read_csv(RAW_DIR / 'asset_inventory.csv')
vulns_raw = pd.read_json(RAW_DIR / 'vulnerability_catalog.json')
{
    'alerts': alerts_raw.shape,
    'assets': assets_raw.shape,
    'vulnerabilities': vulns_raw.shape,
}

{'alerts': (10000, 25), 'assets': (250, 8), 'vulnerabilities': (27, 8)}

In [2]:
summary = []
for name, frame in {'alerts': alerts_raw, 'assets': assets_raw, 'vulnerabilities': vulns_raw}.items():
    summary.append({
        'fuente': name,
        'filas': frame.shape[0],
        'columnas': frame.shape[1],
        'duplicados': frame.duplicated().sum(),
        'nulos_totales': frame.isna().sum().sum(),
    })
pd.DataFrame(summary)

,fuente,filas,columnas,duplicados,nulos_totales
0,alerts,10000,25,0,25055
1,assets,250,8,0,0
2,vulnerabilities,27,8,0,0


In [3]:
pd.concat(
    [
        alerts_raw.dtypes.rename('alerts'),
        assets_raw.dtypes.rename('assets'),
        vulns_raw.dtypes.rename('vulnerabilities'),
    ],
    axis=1,
)

,alerts,assets,vulnerabilities
Timestamp,str,NaN,NaN
Source IP Address,str,NaN,NaN
Destination IP Address,str,NaN,NaN
Source Port,int64,NaN,NaN
Destination Port,int64,NaN,NaN
Protocol,str,NaN,NaN
Packet Length,int64,NaN,NaN
Packet Type,str,NaN,NaN
Traffic Type,str,NaN,NaN
Payload Data,str,NaN,NaN


## 2. Configuracion segura del proyecto

Las variables de entorno utilizadas son `POSTGRES_DB`, `POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_HOST` y `POSTGRES_PORT`. No se deben escribir credenciales directamente en notebooks porque pueden subirse por error a repositorios o compartirse con terceros.

In [4]:
env_check = {
    'POSTGRES_DB': bool(os.getenv('POSTGRES_DB')),
    'POSTGRES_USER': bool(os.getenv('POSTGRES_USER')),
    'POSTGRES_PASSWORD': bool(os.getenv('POSTGRES_PASSWORD')),
    'POSTGRES_HOST': bool(os.getenv('POSTGRES_HOST')),
    'POSTGRES_PORT': bool(os.getenv('POSTGRES_PORT')),
}
env_check

{'POSTGRES_DB': True,
 'POSTGRES_USER': True,
 'POSTGRES_PASSWORD': True,
 'POSTGRES_HOST': True,
 'POSTGRES_PORT': True}

## 3. Funciones de limpieza

Se definen funciones reutilizables para corregir nombres de columnas, estandarizar texto, convertir fechas, tratar nulos, eliminar duplicados y limpiar llaves de relacion.

In [5]:
def to_snake_case(column):
    return column.strip().lower().replace('/', '_').replace('-', '_').replace(' ', '_')

def normalize_text(series):
    return series.astype(str).str.strip().str.replace(r'\\s+', ' ', regex=True)

def clean_alerts(frame):
    df = frame.copy()
    df.columns = [to_snake_case(col) for col in df.columns]
    df = df.drop_duplicates()
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    text_cols = ['protocol', 'traffic_type', 'attack_type', 'severity_level', 'action_taken', 'network_segment']
    for col in text_cols:
        df[col] = normalize_text(df[col])
    df['alerts_warnings'] = df['alerts_warnings'].fillna('No alert')
    df['ids_ips_alerts'] = df['ids_ips_alerts'].fillna('No IDS alert')
    df['proxy_information'] = df['proxy_information'].fillna('No proxy')
    df['machine_id'] = 'SRV-UNTRACKED'
    df['alert_natural_key'] = df['attack_type'].str.upper() + '-' + df['protocol'].str.upper()
    return df

def clean_assets(frame):
    df = frame.copy()
    df.columns = [to_snake_case(col) for col in df.columns]
    df = df.drop_duplicates(subset=['machine_id'])
    for col in ['machine_id', 'server_name', 'operating_system', 'department', 'network_segment', 'criticality']:
        df[col] = normalize_text(df[col])
    return df

def clean_vulnerabilities(frame):
    df = frame.copy()
    df.columns = [to_snake_case(col) for col in df.columns]
    df = df.drop_duplicates(subset=['alert_id'])
    for col in ['attack_type', 'protocol', 'severity_level', 'vulnerability_name', 'remediation_action']:
        df[col] = normalize_text(df[col])
    df['cvss_score'] = pd.to_numeric(df['cvss_score'], errors='coerce').fillna(0)
    return df

## 4. Seleccion de columnas relevantes

Se conservan columnas utiles para analisis SOC: tiempo, IPs, protocolo, tamano de paquete, tipo de ataque, severidad, activo, departamento, criticidad y CVSS. Se eliminan textos largos de payload cuando no aportan al indicador principal.

In [6]:
alerts_clean = clean_alerts(alerts_raw)
assets_clean = clean_assets(assets_raw)
vulns_clean = clean_vulnerabilities(vulns_raw)

ip_to_machine = dict(zip(assets_clean['ip_address'], assets_clean['machine_id']))
alerts_clean['machine_id'] = alerts_clean['destination_ip_address'].map(ip_to_machine).fillna('SRV-UNTRACKED')

alerts_selected = alerts_clean[[
    'timestamp', 'source_ip_address', 'destination_ip_address', 'protocol', 'packet_length',
    'traffic_type', 'anomaly_scores', 'attack_type', 'action_taken', 'severity_level',
    'network_segment', 'machine_id', 'alert_natural_key'
]]
assets_selected = assets_clean[[
    'machine_id', 'server_name', 'operating_system', 'department', 'ip_address',
    'network_segment', 'criticality', 'log_source'
]]
vulns_selected = vulns_clean[[
    'alert_id', 'attack_type', 'protocol', 'severity_level', 'vulnerability_name',
    'cve_code', 'cvss_score', 'remediation_action'
]]
alerts_selected.head(5)

,timestamp,source_ip_address,destination_ip_address,protocol,packet_length,traffic_type,anomaly_scores,attack_type,action_taken,severity_level,network_segment,machine_id,alert_natural_key
0,2023-05-30 06:33:58,103.216.15.12,84.9.164.252,ICMP,503,HTTP,28.67,Malware,Logged,Low,Segment A,SRV-0001,MALWARE-ICMP
1,2020-08-26 07:08:30,78.199.217.198,66.191.137.154,ICMP,1174,HTTP,51.50,Malware,Blocked,Low,Segment B,SRV-0002,MALWARE-ICMP
2,2022-11-13 08:23:25,63.79.210.48,198.219.82.17,UDP,306,HTTP,87.42,DDoS,Ignored,Low,Segment C,SRV-0003,DDOS-UDP
3,2023-07-02 10:38:46,163.42.196.10,101.228.192.255,UDP,385,HTTP,15.79,Malware,Blocked,Medium,Segment B,SRV-0004,MALWARE-UDP
4,2023-07-16 13:11:07,71.166.185.76,189.243.174.238,TCP,1462,DNS,0.52,DDoS,Blocked,Low,Segment C,SRV-0005,DDOS-TCP


## 5. Transformaciones requeridas

Se crean columnas derivadas para analizar picos de ataque por hora y dia, categorizar riesgo y clasificar paquetes de red.

In [7]:
alerts_transformed = alerts_selected.copy()
alerts_transformed['event_hour'] = alerts_transformed['timestamp'].dt.hour
alerts_transformed['event_weekday'] = alerts_transformed['timestamp'].dt.day_name()
alerts_transformed['packet_size_category'] = alerts_transformed['packet_length'].apply(
    lambda value: 'large' if value >= 1000 else ('medium' if value >= 500 else 'small')
)
alerts_transformed['risk_band'] = alerts_transformed['anomaly_scores'].apply(
    lambda score: 'high' if score >= 70 else ('medium' if score >= 40 else 'low')
)

vulns_transformed = vulns_selected.copy()
vulns_transformed['cvss_band'] = vulns_transformed['cvss_score'].apply(
    lambda score: 'critical' if score >= 9 else ('high' if score >= 7 else ('medium' if score >= 4 else 'low'))
)
alerts_transformed.head(5)

,timestamp,source_ip_address,destination_ip_address,protocol,packet_length,traffic_type,anomaly_scores,attack_type,action_taken,severity_level,network_segment,machine_id,alert_natural_key,event_hour,event_weekday,packet_size_category,risk_band
0,2023-05-30 06:33:58,103.216.15.12,84.9.164.252,ICMP,503,HTTP,28.67,Malware,Logged,Low,Segment A,SRV-0001,MALWARE-ICMP,6,Tuesday,medium,low
1,2020-08-26 07:08:30,78.199.217.198,66.191.137.154,ICMP,1174,HTTP,51.50,Malware,Blocked,Low,Segment B,SRV-0002,MALWARE-ICMP,7,Wednesday,large,medium
2,2022-11-13 08:23:25,63.79.210.48,198.219.82.17,UDP,306,HTTP,87.42,DDoS,Ignored,Low,Segment C,SRV-0003,DDOS-UDP,8,Sunday,small,high
3,2023-07-02 10:38:46,163.42.196.10,101.228.192.255,UDP,385,HTTP,15.79,Malware,Blocked,Medium,Segment B,SRV-0004,MALWARE-UDP,10,Sunday,small,low
4,2023-07-16 13:11:07,71.166.185.76,189.243.174.238,TCP,1462,DNS,0.52,DDoS,Blocked,Low,Segment C,SRV-0005,DDOS-TCP,13,Sunday,large,low


## 6. Generacion de identificadores

Se crean identificadores numericos secuenciales para facilitar integracion posterior en modelos dimensionales o cargas a bases de datos.

In [8]:
alerts_transformed = alerts_transformed.reset_index(drop=True)
assets_selected = assets_selected.reset_index(drop=True)
vulns_transformed = vulns_transformed.reset_index(drop=True)

alerts_transformed.insert(0, 'network_alert_id', range(1, len(alerts_transformed) + 1))
assets_selected.insert(0, 'asset_id', range(1, len(assets_selected) + 1))
vulns_transformed.insert(0, 'vulnerability_id', range(1, len(vulns_transformed) + 1))
alerts_transformed[['network_alert_id', 'timestamp', 'machine_id', 'attack_type', 'risk_band']].head(5)

,network_alert_id,timestamp,machine_id,attack_type,risk_band
0,1,2023-05-30 06:33:58,SRV-0001,Malware,low
1,2,2020-08-26 07:08:30,SRV-0002,Malware,medium
2,3,2022-11-13 08:23:25,SRV-0003,DDoS,high
3,4,2023-07-02 10:38:46,SRV-0004,Malware,low
4,5,2023-07-16 13:11:07,SRV-0005,DDoS,low


## 7. Validacion de resultados

Se verifica que los DataFrames transformados no tengan duplicados completos, que las claves principales existan y que los tipos de datos sean consistentes.

In [9]:
validation = []
for name, frame, key in [
    ('alerts_transformed', alerts_transformed, 'network_alert_id'),
    ('assets_selected', assets_selected, 'asset_id'),
    ('vulns_transformed', vulns_transformed, 'vulnerability_id'),
]:
    validation.append({
        'dataframe': name,
        'filas': len(frame),
        'duplicados': frame.duplicated().sum(),
        'nulos_clave': frame[key].isna().sum(),
        'nulos_totales': frame.isna().sum().sum(),
    })
pd.DataFrame(validation)

,dataframe,filas,duplicados,nulos_clave,nulos_totales
0,alerts_transformed,10000,0,0,0
1,assets_selected,250,0,0,0
2,vulns_transformed,27,0,0,0


In [10]:
integrated = (
    alerts_transformed.merge(assets_selected, on='machine_id', how='left')
    .merge(vulns_transformed, on=['attack_type', 'protocol', 'severity_level'], how='left')
)
integrated[['network_alert_id', 'department', 'operating_system', 'attack_type', 'protocol', 'risk_band', 'cvss_score']].head(10)

,network_alert_id,department,operating_system,attack_type,protocol,risk_band,cvss_score
0,1,Tecnologia,Windows,Malware,ICMP,low,4.31
1,2,Finanzas,Windows,Malware,ICMP,medium,4.31
2,3,Operaciones,Windows,DDoS,UDP,high,4.31
3,4,Finanzas,Mac OS X,Malware,UDP,low,6.90
4,5,Operaciones,Windows,DDoS,TCP,low,4.27
5,6,Operaciones,Linux,Malware,UDP,low,6.90
6,7,Tecnologia,Linux,DDoS,TCP,low,9.22
7,8,Tecnologia,Mac OS X,Intrusion,ICMP,medium,9.21
8,9,Finanzas,Mac OS X,Intrusion,TCP,medium,9.21
9,10,Tecnologia,Windows,Malware,UDP,low,6.90


In [11]:
pd.DataFrame({
    'columna': integrated.columns,
    'tipo_dato': [str(dtype) for dtype in integrated.dtypes],
    'nulos': integrated.isna().sum().values,
}).head(30)

,columna,tipo_dato,nulos
0,network_alert_id,int64,0
1,timestamp,datetime64[us],0
2,source_ip_address,str,0
3,destination_ip_address,str,0
4,protocol,str,0
5,packet_length,int64,0
6,traffic_type,str,0
7,anomaly_scores,float64,0
8,attack_type,str,0
9,action_taken,str,0


# Reflexion individual final

Esta semana aprendi que la transformacion de datos es la etapa que convierte fuentes crudas en informacion confiable para analizar. En un contexto de infraestructura y ciberseguridad, los logs pueden venir con fechas mal formateadas, IPs incompletas, textos inconsistentes, duplicados o campos nulos. Si esos problemas no se corrigen, un reporte de incidentes puede mostrar prioridades equivocadas o esconder riesgos reales.

En mi entorno profesional podria aplicar esta practica para normalizar eventos de red, inventarios de servidores, alertas de firewall y catalogos de vulnerabilidades. Usaria funciones de limpieza para estandarizar nombres de columnas, convertir fechas, eliminar duplicados, categorizar severidades y crear identificadores que permitan relacionar fuentes. Documentar cada paso en un notebook es importante porque deja evidencia del criterio usado y permite que otro integrante revise, repita o mejore el proceso. Esto aporta a la toma de decisiones porque ayuda a priorizar parches, detectar segmentos con mas ataques y justificar acciones tecnicas con datos consistentes.